# Pose Control based on Lyapunov Estabilization theory

- Como se faz um projeto não-linear
- A chegada na equação de Lyapunov

## Configuring the environment

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
from coppeliasim_zmqremoteapi_client import RemoteAPIClient
import os

# Parameters of Turtlebot 3
wheel_radius = 0.033
robot_width = 0.287

# Initialize the Remote API
client = RemoteAPIClient()
sim = client.require('sim')

# Open the turtlebot3 scene 
simulation_file = os.getcwd()+'/turtlebot3_pose_estabilization.ttt'
sim.loadScene(simulation_file)

# Use the stepping mode
sim.setStepping(True)   

# Configure the handles
Turtlebot3 = sim.getObjectHandle('/Turtlebot3')
leftMotor = sim.getObjectHandle('/Turtlebot3/left_motor')
rightMotor = sim.getObjectHandle('/Turtlebot3/right_motor')
Goal = sim.getObjectHandle('/ReferenceFrame')

# Error tolerance
min_error = 0.2

# Robot Limits - considering the kinematic model
maxv = 0.26
maxw = 1.82

In [2]:
def draw_robot(x, y, theta, ax, scale=1.0, **kwargs):
    """
    Desenha o contorno do robô no gráfico fornecido.
    
    Parâmetros:
    x, y  : Posição do centro do robô (m)
    theta : Orientação do robô (rad)
    ax    : Objeto Axes do matplotlib onde o robô será desenhado
    scale : Escala do desenho (para ajustar o tamanho visualmente)
    **kwargs: Argumentos opcionais de plotagem (ex: color='b', linestyle='--')
    """
    p = np.zeros((12, 3))
    
    p[:] = [
        [ 1,    1/7,  1/scale],
        [-3/7,  1,    1/scale],
        [-5/7,  6/7,  1/scale],
        [-5/7,  5/7,  1/scale],
        [-3/7,  2/7,  1/scale],
        [-3/7,  0,    1/scale],
        [-3/7, -2/7,  1/scale],
        [-5/7, -5/7,  1/scale],
        [-5/7, -6/7,  1/scale],
        [-3/7, -1,    1/scale],
        [ 1,   -1/7,  1/scale],
        [ 1,    1/7,  1/scale]
    ]
    
    p = scale * p
    
    r = np.array([
        [np.cos(theta),  np.sin(theta)],
        [-np.sin(theta), np.cos(theta)],
        [x,              y]
    ])
    
    p_transf = np.dot(p, r)
    
    X_plot = p_transf[:, 0]
    Y_plot = p_transf[:, 1]
    
    if 'color' not in kwargs and 'c' not in kwargs:
        kwargs['color'] = 'blue'
        
    ax.plot(X_plot, Y_plot, **kwargs)

# Normalize angle to the range [-pi,pi)
def normalize_angle(angle):
    return np.mod(angle+np.pi, 2*np.pi) - np.pi

## Simulation Loop - Aicardi

In [15]:
# Initialize the simulation
sim.startSimulation()

# Control gains
gamma = 1/3
h = 1/9
k = 1/2

while True:

    # Capture the robot pose from simulation
    TBPos=sim.getObjectPosition(Turtlebot3,-1)
    TBXOri=sim.getObjectOrientation(Turtlebot3,-1)
    qTurtlebot = np.array([TBPos[0], TBPos[1], TBXOri[2]])

    # Capture the goal pose
    Goal_Pos = sim.getObjectPosition(Goal,-1)
    Goal_Ori = sim.getObjectOrientation(Goal,-1)
    qGoal = np.array([Goal_Pos[0], Goal_Pos[1],Goal_Ori[2]])

    # Global States
    dx, dy, dth = qGoal - qTurtlebot

    # Transform to Aicardi states
    e = math.sqrt(dx**2 + dy**2)
    alpha = normalize_angle(np.arctan2(dy,dx) - qTurtlebot[2])
    theta_aicardi = normalize_angle(qGoal[2] - np.arctan2(dy,dx))

    # Stopping condition
    if e < min_error:
        # Goal achieved
        print("Alvo alcançado!")
        sim.stopSimulation()
        break

    # Calculate linear and angular velocities from (6) and (9)
    # u = gamma * cos(alpha) * e
    v = gamma * math.cos(alpha) * e
    
    # omega = k*alpha + gamma * (cos(alpha)*sin(alpha)/alpha) * (alpha + h*theta)
    omega = k * alpha + gamma * ((math.cos(alpha) * math.sin(alpha))/alpha) * (alpha + h * theta_aicardi)

    # Saturation Limits
    v = max(min(v, maxv), -maxv)
    omega = max(min(omega, maxw), -maxw) 

    #  DDMR Inverse Kinematics
    w_right = (v + (omega * robot_width / 2)) / wheel_radius
    w_left  = (v - (omega * robot_width / 2)) / wheel_radius

    # Send commands
    sim.setJointTargetVelocity(leftMotor, w_left)
    sim.setJointTargetVelocity(rightMotor, w_right)

    print(f"Erro: {e} | Alpha: {alpha} | Theta: {theta_aicardi}")
    print(f"v: {v}, omega: {omega}\n")
    # Step the simulation
    sim.step()

Erro: 1.7869977616102406 | Alpha: -1.6750276187678086 | Theta: -1.4665650348221768
v: -0.06197466894216595, omega: -0.7996655909808335

Erro: 1.7893218829254793 | Alpha: -1.6350732880114727 | Theta: -1.468058373787219
v: -0.03831099787791264, omega: -0.7940383793487493

Erro: 1.7917323811145291 | Alpha: -1.5946062736762459 | Theta: -1.4691932320916616
v: -0.014219007359741192, omega: -0.7885573002672828

Erro: 1.7942036713884448 | Alpha: -1.5547607017255736 | Theta: -1.4697843495247116
v: 0.009589981448462175, omega: -0.7832859981382695

Erro: 1.7994747488831453 | Alpha: -1.4736325913491388 | Theta: -1.4694854815193348
v: 0.05818956926937134, omega: -0.772566733418587

Erro: 1.801507875549728 | Alpha: -1.4367810118179243 | Theta: -1.4678322427577069
v: 0.08023587015182965, omega: -0.7675396257500794

Erro: 1.8067066170426036 | Alpha: -1.3295099964969292 | Theta: -1.4611253284022774
v: 0.14390532081004961, omega: -0.7515426939291725

Erro: 1.8087628210146183 | Alpha: -1.250548892704101 

## Simulation Loop - Benbouabdallah

In [8]:
# Initialize the Remote API
client = RemoteAPIClient()
sim = client.require('sim')

# Use the stepping mode
sim.setStepping(True)   

# Configure the handles
Turtlebot3 = sim.getObjectHandle('/Turtlebot3')
leftMotor = sim.getObjectHandle('/Turtlebot3/left_motor')
rightMotor = sim.getObjectHandle('/Turtlebot3/right_motor')
Goal = sim.getObjectHandle('/ReferenceFrame')

# Initialize the simulation
sim.startSimulation()

# Control gains
Kv = 2.07
Kw = 1.49

Dd = 0.2

while True:

    # Capture the robot pose from simulation
    TBPos=sim.getObjectPosition(Turtlebot3,-1)
    TBXOri=sim.getObjectOrientation(Turtlebot3,-1)
    qTurtlebot = np.array([TBPos[0], TBPos[1], TBXOri[2]])

    # Capture the goal pose
    Goal_Pos = sim.getObjectPosition(Goal,-1)
    Goal_Ori = sim.getObjectOrientation(Goal,-1)
    qGoal = np.array([Goal_Pos[0], Goal_Pos[1],Goal_Ori[2]])

    # Global States
    dx, dy, dth = qGoal - qTurtlebot
    # Transform to Benbouabdallah states
    D = math.sqrt(dx**2 + dy**2)
    alpha = normalize_angle(-np.arctan2(dy,dx) + qTurtlebot[2])
    theta_aicardi = normalize_angle(qGoal[2] - np.arctan2(dy,dx))
    ed = -Dd + D

    # Stopping condition
    if D < Dd:
        # Goal achieved
        print("Alvo alcançado!")
        sim.stopSimulation()
        break

    # Calculate linear and angular velocities from (14)
    # v_tt = -Kv * ed * math.cos(alpha)
    v_tt = Kv * ed * math.cos(alpha)
    
    # omega = k*alpha + gamma * (cos(alpha)*sin(alpha)/alpha) * (alpha + h*theta)
    omega_tt = -Kw * alpha - (v_tt/D) * math.sin(alpha)

    # Saturation Limits
    # v_tt = max(min(v_tt, maxv), -maxv)
    # omega_tt = max(min(omega_tt, maxw), -maxw) 


    #  DDMR Inverse Kinematics
    w_right = (v_tt + (omega_tt * robot_width / 2)) / wheel_radius
    w_left  = (v_tt - (omega_tt * robot_width / 2)) / wheel_radius

    # Send commands
    sim.setJointTargetVelocity(leftMotor, w_left)
    sim.setJointTargetVelocity(rightMotor, w_right)

    print(f"Erro: {D} | Alpha: {alpha} | Theta: {theta_aicardi}")
    print(f"v: {v_tt}, omega: {omega_tt}\n")
    # Step the simulation
    sim.step()

Erro: 1.7869977616102406 | Alpha: 1.6750276187678086 | Theta: -1.4665650348221768
v: -0.3417890314325766, omega: -2.3055647883531445

Erro: 1.7905453400626357 | Alpha: 1.6045981572967225 | Theta: -1.4711661286363698
v: -0.11126893059407188, omega: -2.3287442704075225

Erro: 1.7982714122943122 | Alpha: 1.4789484794673537 | Theta: -1.474916538904759
v: 0.30344435812094994, omega: -2.3716642241610453

Erro: 1.8050895601467312 | Alpha: 1.3639277902141718 | Theta: -1.4714767652560217
v: 0.6824361923256023, omega: -2.4022539401669465

Erro: 1.8062602234471883 | Alpha: 1.2630389034626326 | Theta: -1.4569209547913449
v: 1.0072037600383295, omega: -2.4133467357227767

Erro: 1.7988524393997694 | Alpha: 1.166422169461029 | Theta: -1.4322468528900263
v: 1.302150166345033, omega: -2.4034656462425232

Erro: 1.7819749470803479 | Alpha: 1.0767507240643432 | Theta: -1.3998184238614488
v: 1.5528295934905214, omega: -2.371566242715544

Erro: 1.75274746717977 | Alpha: 0.9966103198245522 | Theta: -1.361087